In [2]:
import os
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer

print("1. 加载 Tokenizer...")
# 保证使用与学生/教师模型一致的 Tokenizer
tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen2.5-1.5B")

print("2. 加载 Alpaca 数据集...")
raw_dataset = load_dataset("yahma/alpaca-cleaned", split="train[:2200]")

alpaca_prompt_without_answer = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
"""

def format_and_compute_length(example):
    # 处理 input，兼容 input 为空字符串的情况
    input_text = example["input"] if example["input"] else ""
    
    # 格式化拼接出完整的 Prompt
    text = alpaca_prompt_without_answer.format(example["instruction"], input_text)
    
    # 计算 Token 长度
    tokens = tokenizer(text, truncation=False)
    return {"text": text, "length": len(tokens["input_ids"])}

print("3. 正在格式化并计算 Token 长度...")
dataset_with_length = raw_dataset.map(format_and_compute_length, num_proc=8)

# 隔离出固定测试集 (最后 200 条)，其余为训练集
test_dataset = dataset_with_length.select(range(2000, 2200))
train_dataset = dataset_with_length.select(range(2000))

# 基于数据分布自动寻找最佳阈值
lengths = train_dataset["length"]
p25 = int(np.percentile(lengths, 25))
p50 = int(np.percentile(lengths, 50))
p75 = int(np.percentile(lengths, 75))

print("\n📊 训练集 Token 长度统计：")
print(f"最短: {min(lengths)} | 最长: {max(lengths)} | 平均: {int(np.mean(lengths))}")
print(f"25% 分位数 (Easy 阈值): <= {p25}")
print(f"50% 分位数 (Medium 阈值): <= {p50}")
print(f"75% 分位数 (Hard 阈值): <= {p75}")

print("\n4. 开始按照分位数物理切分数据集并落盘...")
# 阶段一 (Easy)
easy_dataset = train_dataset.filter(lambda x: x["length"] <= p25, num_proc=8)
# 阶段二 (Medium)
medium_dataset = train_dataset.filter(lambda x: x["length"] <= p50, num_proc=8)
# 阶段三 (Hard)
hard_dataset = train_dataset.filter(lambda x: x["length"] <= p75, num_proc=8)
# 阶段四 (Full)
full_dataset = train_dataset

os.makedirs("./data_splits", exist_ok=True)
test_dataset.save_to_disk("./data_splits/data_test")
easy_dataset.save_to_disk("./data_splits/data_easy")
medium_dataset.save_to_disk("./data_splits/data_medium")
hard_dataset.save_to_disk("./data_splits/data_hard")
full_dataset.save_to_disk("./data_splits/data_full")

print("\n 切分完成！各阶段数据量统计：")
print(f"Test Set: {len(test_dataset)} 条")
print(f"Easy:     {len(easy_dataset)} 条 (阈值 {p25})")
print(f"Medium:   {len(medium_dataset)} 条 (阈值 {p50})")
print(f"Hard:     {len(hard_dataset)} 条 (阈值 {p75})")
print(f"Full:     {len(full_dataset)} 条")

1. 加载 Tokenizer...
2. 加载 Alpaca 数据集...
3. 正在格式化并计算 Token 长度...

📊 训练集 Token 长度统计：
最短: 41 | 最长: 506 | 平均: 57
25% 分位数 (Easy 阈值): <= 46
50% 分位数 (Medium 阈值): <= 50
75% 分位数 (Hard 阈值): <= 57

4. 开始按照分位数物理切分数据集并落盘...


Saving the dataset (0/1 shards):   0%|          | 0/200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/522 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1022 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1500 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2000 [00:00<?, ? examples/s]


 切分完成！各阶段数据量统计：
Test Set: 200 条
Easy:     522 条 (阈值 46)
Medium:   1022 条 (阈值 50)
Hard:     1500 条 (阈值 57)
Full:     2000 条
